<a href="https://colab.research.google.com/github/svyatoslavna/nlp_hw/blob/main/rnn_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: генерация текста с помощью RNN (собственный корпус)

## Задача

1. Собрать свой небольшой текстовый корпус (не менее 1000 предложений) с помощью библиотеки `requests` (парсинг новостного сайта, блога, форума и т.д.).
2. Обучить рекуррентную нейросеть (RNN/LSTM) на собранных данных для генерации текста (по образцу из приложенного ноутбука `Copy_of_rnn.ipynb`).
3. После обучения вывести на экран 2–3 сгенерированных предложения.
4. *Дополнительно (на 5 баллов, но не обязательно):* посчитать метрику перплексии (perplexity) на валидационной выборке.
5. *Для себя (не оценивается):* обучить модель с использованием GPU.

## Критерии оценки

- **3 балла** — корпус собран (≥1000 предложений), модель обучена, сгенерировано хотя бы 1 предложение.
- **4 балла** — всё из п.3 + код с комментариями, объясняющими ключевые этапы.
- **5 баллов** — всё из п.4 + дополнительно посчитана метрика перплексии.

## Важно

- Качество сгенерированного текста не оценивается.
- Выберите **свой уникальный сайт** для парсинга и укажите его в отчёте.
- Не используйте готовые датасеты из интернета.


## 1. Установка и импорт библиотек

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re
import pandas as pd

In [ ]:
!pip install beautifulsoup4 requests tensorflow lxml -q

import requests
from bs4 import BeautifulSoup
import time
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow версия:", tf.__version__)
print("GPU доступна:", tf.config.list_physical_devices('GPU'))

TensorFlow версия: 2.20.0
GPU доступна: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Парсинг текстового корпуса

**Ваш уникальный сайт:** [https://www.avito.ru/](https://www.avito.ru/)

**Обоснование выбора:** Для генерации объявлений о продаже каких-либо прдеметов (заголовок + описание).

Эту часть делала в Jupyter Notebook, т.к. ip-адрес Colab'а сайт блокировал, с первого запроса - ошибка 429. В Jupyter Notebook с моим ip всё заработало без проблем - код 200.

In [ ]:
def scrape_corpus(base_url, num_pages=5):
    """
    Парсинг текстового корпуса с выбранного сайта.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    texts = []

    for page in range(1, num_pages + 1):
        if page == 1:
            url = base_url
        else:
            url = f"{base_url}?p={page}"

        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            # находим все блоки объявлений
            items = soup.find_all('div', attrs={'data-marker': 'item'})

            print(f"Страница {page}: найдено {len(items)} объявлений")

            for item in items:
                # заголовок
                title_elem = item.find('a', attrs={'data-marker': 'item-title'})
                title = title_elem.text.strip() if title_elem else ''

                # цена
                price_meta = item.find('meta', attrs={'itemprop': 'price'})
                price = price_meta.get('content') if price_meta else ''

                # описание (но оно на страничке не полностью отражается:((((()
                desc_meta = item.find('meta', attrs={'itemprop': 'description'})
                description = desc_meta.get('content') if desc_meta else ''

                # сохраняем только если есть заголовок
                if title:
                    texts.append({
                        'title': title,
                        'price': price,
                        'description': description
                    })

            print(f"  Всего собрано за всё время: {len(texts)} записей")

            time.sleep(3)

        except Exception as e:
            print(f"Ошибка на странице {page}: {e}")

    return texts

In [ ]:
# список страничек с разными категориями товаров

categories = {'Игрушки и товары для детей': 'https://www.avito.ru/sankt-peterburg/tovary_dlya_detey_i_igrushki',
            'Готовый бизнес и оборудование для бизнеса': 'https://www.avito.ru/sankt-peterburg/dlya_biznesa',
            'Женская одежда': 'https://www.avito.ru/sankt-peterburg/odezhda_obuv_aksessuary/zhenskaya_odezhda-ASgBAgICAUTeAtYL',
            'Аксессуары': 'https://www.avito.ru/sankt-peterburg/odezhda_obuv_aksessuary/kupit-aksessuary-ASgBAgICAUTeAtoL',
            'Сумки, рюкзаки и чемоданы': 'https://www.avito.ru/sankt-peterburg/odezhda_obuv_aksessuary/sumki_ryukzaki_i_chemodany-ASgBAgICAUTeArip1gI',
            'Хобби и отдых': 'https://www.avito.ru/sankt-peterburg/hobbi_i_otdyh',
            'Красота и здоровье': 'https://www.avito.ru/sankt-peterburg/krasota_i_zdorove'}

corpus = {} # сюда будем собирать все данные

for cat_name, cat_url in categories.items():
    corpus[cat_name] = scrape_corpus(cat_url, num_pages=3)
    print(f"Всего собрано в категории '{cat_name}': {len(corpus[cat_name])}")
    time.sleep(5)

Страница 1: найдено 81 объявлений
  Всего собрано за всё время: 81 записей
Страница 2: найдено 50 объявлений
  Всего собрано за всё время: 131 записей
Страница 3: найдено 50 объявлений
  Всего собрано за всё время: 181 записей
Всего собрано в категории 'Игрушки и товары для детей': 181
Страница 1: найдено 50 объявлений
  Всего собрано за всё время: 50 записей
Страница 2: найдено 50 объявлений
  Всего собрано за всё время: 100 записей
Страница 3: найдено 50 объявлений
  Всего собрано за всё время: 150 записей
Всего собрано в категории 'Готовый бизнес и оборудование для бизнеса': 150
Страница 1: найдено 81 объявлений
  Всего собрано за всё время: 81 записей
Страница 2: найдено 50 объявлений
  Всего собрано за всё время: 131 записей
Страница 3: найдено 50 объявлений
  Всего собрано за всё время: 181 записей
Всего собрано в категории 'Женская одежда': 181
Страница 1: найдено 81 объявлений
  Всего собрано за всё время: 81 записей
Страница 2: найдено 50 объявлений
  Всего собрано за всё врем

In [ ]:
total = 0
for cat_name, items in corpus.items():
    total += len(items)

print(f"Всего объявлений по всем категориям: {total}")

Всего объявлений по всем категориям: 1169


In [ ]:
# оформляем датасет

dataset = []

for cat_name, items in corpus.items():
    for item in items:
        dataset.append({
            'category': cat_name,
            'title': item['title'],
            'price': item['price'],
            'description': item['description']
        })

df = pd.DataFrame(dataset)

print(f"Датасет создан: {len(df)} строк")
print(f"Колонки: {df.columns.tolist()}")

Датасет создан: 1169 строк
Колонки: ['category', 'title', 'price', 'description']


In [ ]:
df.to_csv('avito_dataset.csv', index=False, encoding='utf-8') # сохраняем файл

## 3. Подготовка данных для RNN

Токенизация, создание последовательностей, паддинг.

In [ ]:
df = pd.read_csv('avito_dataset.csv') # выгружаем датасет
df

,category,title,price,description
0,Игрушки и товары для детей,Шкаф в детскую,10099,Шкаф в детскую на заказ.\n\nБесплатно:\n\nДост...
1,Игрушки и товары для детей,"Пододеяльник 1,5 сп.Ромб. Бязь ГОСТ 100%хлопок",455,Пододеяльник с ромбом 145х210.
2,Игрушки и товары для детей,Шкаф в детскую на заказ,9900,Мебель для дома по индивидуальным проектам под...
3,Игрушки и товары для детей,Стильный шкаф в детскую,9090,Шкаф на заказ.\n\nБесплатно.\n\nДоставляем!\n\...
4,Игрушки и товары для детей,Шкаф на заказ в детскую,9900,Мебель для дома по индивидуальным проектам под...
...,...,...,...,...
1164,Красота и здоровье,Аренда коляски инвалидной с электроприводом,1000,Если вам нужна максимальная свобода в передвиж...
1165,Красота и здоровье,Liquides Imaginaires Blanche Bete распив,2600,Аромат из личной коллекции.\n\nLes Liquides Im...
1166,Красота и здоровье,Бахилы одноразовые оптом,79,Цены ниже рынка! Скидки для ИП и Юрлиц. 2000+ ...
1167,Красота и здоровье,Легендарные кремы французской аптеки,2650,"Faq:\nАкне у подростков, пигментация, омоложен..."


In [ ]:
# лучше поменять название колонки, чтобы была понятна валюта

df = df.rename(columns={'price': 'price_rub'})
df

,category,title,price_rub,description
0,Игрушки и товары для детей,Шкаф в детскую,10099,Шкаф в детскую на заказ.\n\nБесплатно:\n\nДост...
1,Игрушки и товары для детей,"Пододеяльник 1,5 сп.Ромб. Бязь ГОСТ 100%хлопок",455,Пододеяльник с ромбом 145х210.
2,Игрушки и товары для детей,Шкаф в детскую на заказ,9900,Мебель для дома по индивидуальным проектам под...
3,Игрушки и товары для детей,Стильный шкаф в детскую,9090,Шкаф на заказ.\n\nБесплатно.\n\nДоставляем!\n\...
4,Игрушки и товары для детей,Шкаф на заказ в детскую,9900,Мебель для дома по индивидуальным проектам под...
...,...,...,...,...
1164,Красота и здоровье,Аренда коляски инвалидной с электроприводом,1000,Если вам нужна максимальная свобода в передвиж...
1165,Красота и здоровье,Liquides Imaginaires Blanche Bete распив,2600,Аромат из личной коллекции.\n\nLes Liquides Im...
1166,Красота и здоровье,Бахилы одноразовые оптом,79,Цены ниже рынка! Скидки для ИП и Юрлиц. 2000+ ...
1167,Красота и здоровье,Легендарные кремы французской аптеки,2650,"Faq:\nАкне у подростков, пигментация, омоложен..."


In [ ]:
# подготавливаю тексты по шаблону "заголовок: описание", вместо None - пустая строка

texts = df['title'].fillna('') + ': ' + df['description'].fillna('')
texts = texts.tolist()

# создаём токенизатор
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

# преобразуем тексты в последовательности чисел
sequences = tokenizer.texts_to_sequences(texts)

# создаём входные и выходные данные для causal language modeling
X, y = [], []
for seq in sequences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

# паддинг
X = pad_sequences(X)

# one-hot encoding для y
vocab_size = len(tokenizer.word_index) + 1
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

print(f"Размер входных данных X: {X.shape}")
print(f"Размер выходных данных y: {y.shape}")
print(f"Размер словаря: {vocab_size}")

Размер входных данных X: (40165, 56)
Размер выходных данных y: (40165, 10723)
Размер словаря: 10723


## 4. Создание и обучение модели RNN (LSTM)

Для выполнения этой части из Jupyter Notebook перешла в Google Colba, чтобы использовать его мощности и обучать модель на GPU.

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=100, input_length=X.shape[1]))
model.add(LSTM(150, return_sequences=False))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# обучение (увеличивать эпохи не стану, т.к. модель, похоже, и так переобучается)
history = model.fit(X, y, epochs=10, batch_size=32, validation_split=0.2)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 19s 13ms/step - accuracy: 0.0324 - loss: 8.2037 - val_accuracy: 0.0326 - val_loss: 8.9573
Epoch 2/10
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - accuracy: 0.0435 - loss: 7.2729 - val_accuracy: 0.0360 - val_loss: 9.3588
Epoch 3/10
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - accuracy: 0.0878 - loss: 6.4696 - val_accuracy: 0.0301 - val_loss: 9.8805
Epoch 4/10
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - accuracy: 0.1385 - loss: 5.6508 - val_accuracy: 0.0275 - val_loss: 10.3485
Epoch 5/10
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - accuracy: 0.1974 - loss: 4.8802 - val_accuracy: 0.0233 - val_loss: 10.7552
Epoch 6/10
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.2629 - loss: 4.1802 - val_accuracy: 0.0251 - val_loss: 11.2565
Epoch 7/10
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - accuracy: 0.3408 - loss: 3.5559 - val_accuracy: 0.0222 - val_loss: 11.6587
Epoch 8/10
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.43

## 5. Генерация текста

Функция генерации и вывод 2–3 предложений.

In [ ]:
def generate_text(seed_text, next_words, max_sequence_len):
    """
    Функция для генерации текста обученной моделью
    """
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

In [ ]:
# пример генерации 1
seed = "Шкаф"  # замените на любое начало
generated = generate_text(seed, next_words=10, max_sequence_len=X.shape[1])
print("Сгенерированный текст:", generated)

# пример 2
seed2 = "Детский"
generated2 = generate_text(seed2, next_words=10, max_sequence_len=X.shape[1])
print("Сгенерированный текст 2:", generated2)

# пример 3
seed2 = "Женские"
generated2 = generate_text(seed2, next_words=10, max_sequence_len=X.shape[1])
print("Сгенерированный текст 3:", generated2)

# если представить, что это объявления на товары из Китая, то правдоподобно получается

Сгенерированный текст: Шкаф холодильный polair cc214 в изысканном доставка состоянии в наличии напишите
Сгенерированный текст 2: Детский комплекс детская игровая площадка детская игровая площадка детский игровой комплекс
Сгенерированный текст 3: Женские на аккумуляторе дeтcкий мотоцикл oгpомный выбор детcких мотоциклов большoй шoу


## 6. (Дополнительно, на 5 баллов) Расчёт перплексии

Перплексия = exp(loss). Чем ниже, тем лучше модель предсказывает последовательность.

In [ ]:
# оцениваем модель на валидационных данных
loss, accuracy = model.evaluate(X, y, verbose=0)
perplexity = np.exp(loss)

print(f"Потери (loss): {loss:.4f}")
print(f"Перплексия: {perplexity:.4f}")
print("Пояснение: перплексия показывает, насколько модель 'удивлена' тестовыми данными. Чем ниже — тем лучше.")

Потери (loss): 3.7526
Перплексия: 42.6323
Пояснение: перплексия показывает, насколько модель 'удивлена' тестовыми данными. Чем ниже — тем лучше.


## 7. GPU (не оценивается)

Убедитесь, что обучение запускалось на GPU:

In [ ]:
print("GPU доступна:", tf.config.list_physical_devices('GPU')) # обучалась на GPU

GPU доступна: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


В Google Colab: Среда выполнения - Изменить тип среды выполнения - Выберите T4 GPU.

Документация: https://www.tensorflow.org/guide/gpu

In [ ]:
print("Устройства GPU:", tf.config.list_physical_devices('GPU'))

Устройства GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
